In [ ]:
# Diffusion Transformers (DiT) for CelebA-HQ
#
# Implements a Diffusion Transformer from scratch:
#   - forward diffusion (noise schedule + noising)
#   - timestep embeddings
#   - DiT blocks: patchify, adaptive LayerNorm (adaLN), self-attention, MLP
#   - denoising objective (p_losses) and DDPM sampling
#   - end-to-end training + sample generation on CelebA-HQ faces
#
# EXPECTED wall-time on one Kaggle T4 GPU (details in each cell):
#   imports + config           : ~5 s
#   dataset scan + loader      : ~10-30 s (reads 30k HQ images, cached in RAM)
#   forward diffusion demo     : ~5 s
#   model forward check        : ~5 s
#   untrained sample sanity    : ~10-20 s
#   training (20 epochs)       : ~25-40 min total
#   loss curve + 8x8 grid      : ~30-60 s

import math
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

In [ ]:
# ---- Configuration (DiT-small / ~28M params) ----
class Config:
    # data
    dataset = 'CelebA-HQ'  # high-quality, high-res face images
    resolution = 64        # HQ input (1024) is center-cropped + resized to 64
    in_channels = 3        # RGB color
    batch_size = 64
    num_workers = 4

    # DiT architecture
    patch_size = 8         # each patch is patch_size x patch_size pixels
    hidden_size = 192      # d_model
    n_heads = 4            # attention heads
    depth = 6              # number of DiT blocks
    mlp_ratio = 4.0        # hidden dim of the MLP block

    # diffusion
    num_timesteps = 1000   # T
    beta_start = 1e-4      # linear noise schedule (fallback)
    beta_end = 0.02

    # training
    epochs = 20
    lr = 1e-3
    weight_decay = 0.0
    warmup_steps = 200
    use_amp = True
    log_every = 200        # print loss every N iters
    sample_every_n_epochs = 5
    seed = 0


cfg = Config()

# resolution is chosen to divide evenly by patch_size (64 / 8 = 8 -> 64 patches)
IMG_SIZE = cfg.resolution

torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)

In [ ]:
# ---- Dataset: CelebA-HQ (from /kaggle/input) ----
# Auto-detects whichever CelebA-HQ variant is attached to the notebook.
# Most variants ship the images under a "data/" folder containing one
# subfolder per image class (e.g. "000001".."029999" or 0..9 buckets).
def find_image_root(root):
    for dirpath, dirnames, filenames in os.walk(root):
        # a folder that itself holds image files is a usable image root
        if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in filenames):
            return dirpath
    return None


def load_images(data_root):
    """Walk a CelebA-HQ dataset and return (paths, class_labels) for every image."""
    imgs, labels = [], []
    exts = ('.jpg', '.jpeg', '.png', '.bmp')
    for cls_idx, (dirpath, _, filenames) in enumerate(sorted(os.walk(data_root))):
        for f in sorted(filenames):
            if f.lower().endswith(exts):
                imgs.append(os.path.join(dirpath, f))
                labels.append(cls_idx)
    return imgs, labels


if os.path.exists('/kaggle/input'):
    root = '/kaggle/input'
    subdirs = sorted(os.listdir(root))
    print('Kaggle input datasets found:', subdirs)
    # search each attached dataset for an image root
    root_imgs = None
    for sub in subdirs:
        candidate = find_image_root(os.path.join(root, sub))
        if candidate is not None:
            root_imgs = candidate
            break
    if root_imgs is None:
        raise RuntimeError('Could not find an image folder in /kaggle/input. '
                           'Attach a CelebA-HQ style dataset (folders of images).')
    image_paths, class_labels = load_images(root_imgs)
    print('Using dataset folder:', root_imgs)
else:  # local fallback for quick experiments
    fallback = './celebahq_local'
    if os.path.isdir(fallback):
        image_paths, class_labels = load_images(fallback)
    else:
        raise RuntimeError('No /kaggle/input and no local ./celebahq_local folder. '
                           'On Kaggle, attach a CelebA-HQ dataset to the notebook.')

print('Total images:', len(image_paths))
assert len(image_paths) > 0, 'Dataset is empty.'

# A tiny (and fast) Dataset that returns the pre-resized tensor for each path.
class ImageFolderDS(torch.utils.data.Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.transform(img)


from PIL import Image

transform = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.10)),      # upsample slightly...
    transforms.CenterCrop(IMG_SIZE),              # ...center-crop to a clean square
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # [-1, 1]
])

train_ds = ImageFolderDS(image_paths, transform)
train_dl = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                      num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
print('Image shape after transform:', tuple(next(iter(train_dl))[0].shape))
# Estimated time: <1 min to build the index + loader

## 1. Forward diffusion

We pre-compute the cumulative products `alpha_bar_t` that let us sample a noisy image at any timestep directly:

$$
q(x_t \mid x_0) = \mathcal{N}(x_t; \;\sqrt{\bar\alpha_t}\,x_0,\,(1-\bar\alpha_t)I)
$$

or equivalently `x_t = sqrt(alpha_bar) * x_0 + sqrt(1 - alpha_bar) * eps`.

The beta schedule sets the process — we use **cosine** (nicer visual quality than linear), with the DDPM-style **linear** schedule included for comparison.

In [ ]:
# ---- Beta / alpha-cumprod schedules ----
def cosine_beta_schedule(timesteps, s=0.008):
    """Cosine noise schedule (better visual quality than linear)."""
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.9999)


def linear_beta_schedule(timesteps, beta_start=cfg.beta_start, beta_end=cfg.beta_end):
    return torch.linspace(beta_start, beta_end, timesteps)


def get_noise_schedule(timesteps, kind='cosine'):
    if kind == 'linear':
        betas = linear_beta_schedule(timesteps)
    elif kind == 'cosine':
        betas = cosine_beta_schedule(timesteps)
    else:
        raise ValueError(kind)

    alphas = 1. - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)

    return {
        'betas': betas,
        'alphas': alphas,
        'alphas_cumprod': alphas_cumprod,
        'sqrt_alphas_cumprod': torch.sqrt(alphas_cumprod),
        'sqrt_one_minus_alphas_cumprod': torch.sqrt(1. - alphas_cumprod),
    }


sch = get_noise_schedule(cfg.num_timesteps, kind='cosine')
print('beta range: %.5f .. %.5f' % (sch['betas'].min().item(), sch['betas'].max().item()))
# Estimated time: <1 s on GPU

In [ ]:
# ---- Forward noising utility -----
def q_sample(x0, t, noise=None):
    """Sample x_t ~ q(x_t | x_0). x0 in [-1,1], t in [0, T)."""
    if noise is None:
        noise = torch.randn_like(x0)
    sqrt_abar = sch['sqrt_alphas_cumprod'].to(x0.device)[t].view(-1, 1, 1, 1)
    sqrt_one_minus_abar = sch['sqrt_one_minus_alphas_cumprod'].to(x0.device)[t].view(-1, 1, 1, 1)
    return sqrt_abar * x0 + sqrt_one_minus_abar * noise, noise


def show_img(img, ax):
    """Render a [-1,1] (C,H,W) tensor as a color image."""
    img = img.detach().cpu().clamp(-1, 1).numpy().transpose(1, 2, 0)
    ax.imshow((img + 1) / 2)   # map [-1,1] -> [0,1]
    ax.axis('off')


# Demo: visualize one image at progressively louder noise levels
demo_x, _ = next(iter(train_dl))
demo_img = demo_x[0:1]
ts = [0, 100, 300, 500, 700, 999]
fig, axes = plt.subplots(1, len(ts), figsize=(2 * len(ts), 2.2))
for ax, t in zip(axes, ts):
    x_t, _ = q_sample(demo_img, torch.full((1,), t, dtype=torch.long))
    show_img(x_t[0], ax)
    ax.set_title('t=%d' % t)
plt.suptitle('Forward diffusion (increasing noise)')
plt.tight_layout()
plt.show()
# Estimated time: ~2 s on GPU

## 2. Timestep embeddings

Following Vaswani et al., timesteps are embedded with sinusoidal positional embeddings and fed through a small MLP to produce the conditioning vector used by every DiT block (adiabatic to the query length).

In [ ]:
def timestep_embedding(t, dim, max_period=10000):
    """Sinusoidal embedding of timesteps (Vaswani et al. 2017)."""
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half, dtype=torch.float32) / half)
    freqs = freqs.to(t.device)
    args = t.float()[:, None] * freqs[None, :]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)  # (B, dim)


class TimestepEmbedder(nn.Module):
    """Maps a timestep to a learned conditioning vector of size hidden_size."""
    def __init__(self, hidden_size, frequency_embedding_size=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(frequency_embedding_size, hidden_size, bias=True),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size, bias=True),
        )
        self.frequency_embedding_size = frequency_embedding_size

    def forward(self, t):
        t_freq = timestep_embedding(t, self.frequency_embedding_size)
        return self.mlp(t_freq)

## 3. DiT blocks

- **Patchify**: split the image into non-overlapping `patch_size` x `patch_size` patches and map each into `hidden_size` via a conv layer (`nn.Conv2d`). A CelebA-HQ face resized to 64x64 with `patch_size=8` gives `(64/8)^2 = 64` patches -> a sequence of 64 tokens.
- **DiTBlock**: `adaLN`-modulated LayerNorm predicts scale `g`, shift `b`, and gate `alpha` for the [attention, MLP] pre-norms; each is added into a residual self-attention + MLP.
- **Unpatchify**: combine the per-patch embeddings back into the pixel grid and down-project to `in_channels`.

In [ ]:
class Attention(nn.Module):
    """Multi-head self-attention over the sequence of patches."""
    def __init__(self, hidden_size, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = hidden_size // n_heads
        self.qkv = nn.Linear(hidden_size, 3 * hidden_size, bias=False)
        self.proj = nn.Linear(hidden_size, hidden_size, bias=True)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        x = F.scaled_dot_product_attention(q, k, v)  # (B, H, N, head_dim)
        x = x.transpose(1, 2).reshape(B, N, C)
        return self.proj(x)


class Mlp(nn.Module):
    def __init__(self, hidden_size, hidden_size_mlp):
        super().__init__()
        self.fc1 = nn.Linear(hidden_size, hidden_size_mlp, bias=True)
        self.act = nn.GELU(approximate='tanh')
        self.fc2 = nn.Linear(hidden_size_mlp, hidden_size, bias=True)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class DiTBlock(nn.Module):
    """Transformer block with adaptive LayerNorm (adaLN) conditioning."""
    def __init__(self, hidden_size, n_heads, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.attn = Attention(hidden_size, n_heads)
        self.norm2 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.mlp = Mlp(hidden_size, int(hidden_size * mlp_ratio))
        # 6 parameters per block (gamma1, beta1, alpha1, gamma2, beta2, alpha2)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, 6 * hidden_size, bias=True),
        )

    def forward(self, x, c):
        mod = self.adaLN_modulation(c)                 # (B, 6*C)
        g1, b1, a1, g2, b2, a2 = mod.chunk(6, dim=-1)

        x = x + a1.unsqueeze(1) * self.attn(
            (1 + g1.unsqueeze(1)) * self.norm1(x) + b1.unsqueeze(1))
        x = x + a2.unsqueeze(1) * self.mlp(
            (1 + g2.unsqueeze(1)) * self.norm2(x) + b2.unsqueeze(1))
        return x


class FinalLayer(nn.Module):
    """Conditioned LayerNorm + linear back to patch_size^2 * in_channels pixels."""
    def __init__(self, hidden_size, patch_size, out_channels):
        super().__init__()
        self.norm_final = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.linear = nn.Linear(hidden_size, patch_size * patch_size * out_channels, bias=True)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, 2 * hidden_size, bias=True),
        )

    def forward(self, x, c):
        g, b = self.adaLN_modulation(c).chunk(2, dim=-1)
        return self.linear(
            (1 + g.unsqueeze(1)) * self.norm_final(x) + b.unsqueeze(1))


class DiT(nn.Module):
    """Diffusion Transformer (Peebles & Xie, 2023)."""
    def __init__(self, args=None):
        super().__init__()
        a = args or cfg
        self.in_channels = a.in_channels
        self.patch_size = a.patch_size
        self.hidden_size = a.hidden_size
        self.out_channels = a.in_channels
        self.n_heads = a.n_heads
        self.depth = a.depth
        self.num_timesteps = a.num_timesteps

        # input/positional encoding
        self.x_embedder = nn.Conv2d(a.in_channels, a.hidden_size,
                                    kernel_size=a.patch_size, stride=a.patch_size)
        self.t_embedder = TimestepEmbedder(a.hidden_size)

        self.pos_embed = nn.Parameter(
            torch.zeros(1, (IMG_SIZE // a.patch_size) ** 2, a.hidden_size))
        nn.init.normal_(self.pos_embed, std=0.02)

        self.blocks = nn.ModuleList([
            DiTBlock(a.hidden_size, a.n_heads, a.mlp_ratio) for _ in range(a.depth)
        ])
        self.final_layer = FinalLayer(a.hidden_size, a.patch_size, self.out_channels)
        self.initialize_weights()

    def initialize_weights(self):
        def _basic_init(module):
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
        self.apply(_basic_init)
        nn.init.xavier_uniform_(self.x_embedder.weight, gain=0.02)
        nn.init.constant_(self.x_embedder.bias, 0)
        for block in self.blocks:
            nn.init.constant_(block.adaLN_modulation[-1].weight, 0)
            nn.init.constant_(block.adaLN_modulation[-1].bias, 0)
        nn.init.constant_(self.final_layer.adaLN_modulation[-1].weight, 0)
        nn.init.constant_(self.final_layer.adaLN_modulation[-1].bias, 0)
        nn.init.constant_(self.final_layer.linear.weight, 0)
        nn.init.constant_(self.final_layer.linear.bias, 0)

    def unpatchify(self, x):
        """x: (B, N, patch_size^2*C) -> (B, C, H, W)"""
        c = self.out_channels
        p = self.patch_size
        h = w = IMG_SIZE // p
        x = x.reshape(shape=(x.shape[0], c, h, p, w, p))
        x = torch.einsum('bchpwq->bhwpqc', x)
        return x.reshape(shape=(x.shape[0], c, h * p, w * p))

    def forward(self, x, t):
        """x: (B, C, H, W) noisy image; t: (B,) integer timesteps."""
        x = self.x_embedder(x)                # (B, C, H/p, W/p)
        x = x.flatten(2).transpose(1, 2)      # (B, N, C)
        x = x + self.pos_embed                # positional embedding
        t = self.t_embedder(t)                # (B, C)
        for block in self.blocks:
            x = block(x, t)
        x = self.final_layer(x, t)            # (B, N, p^2*C)
        return self.unpatchify(x)             # (B, C, H, W)

## 4. Training objective

The model is trained to predict the Gaussian noise `eps` added at `time t` (the standard simplified DDPM loss):

$$
L = \mathbb{E}_{t, \, x_0, \, \varepsilon} \big[ \| \varepsilon - \varepsilon_\theta(x_t, t) \|^2 \big]
$$

With `use_amp=True` we keep the noise schedule and loss statistics in fp32 and run the forward pass in fp16 via the AMP autocaster, as in the original DiT release.

**Expected time:** model is built and a forward/backward sanity check runs in ~5 s on a T4.

In [ ]:
def p_losses(model, x0, t, noise=None, use_amp=True):
    """Standard DDPM denoising loss: predict the noise added at time t."""
    noise = torch.randn_like(x0) if noise is None else noise
    x_t, _ = q_sample(x0, t, noise)
    with autocast(enabled=use_amp):
        pred = model(x_t, t)
        return (pred - noise).square().mean()


model = DiT().to(device)
n_params = sum(p.numel() for p in model.parameters())
print('DiT parameter count: {:,}'.format(n_params))

# quick forward-pass sanity check
xb, _ = next(iter(train_dl))
xb = xb[:2].to(device)
tb = torch.randint(0, cfg.num_timesteps, (2,), device=device)
out = model(xb, tb)
loss = p_losses(model, xb, tb)
print('Forward out shape:', tuple(out.shape))
print('Loss:', float(loss))
# Estimated time: ~5 s (T4)

In [ ]:
# ---- Sampling (DDPM reverse process) ----
@torch.no_grad()
def sample(model, n=16, steps=cfg.num_timesteps, device='cpu'):
    """Sample images from a trained model using the DDPM reverse process."""
    model.eval()
    betas = sch['betas'].to(device)
    alphas = 1. - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)

    x = torch.randn(n, cfg.in_channels, IMG_SIZE, IMG_SIZE, device=device)

    for i in range(steps - 1, -1, -1):
        t_batched = torch.full((n,), i, dtype=torch.long, device=device)
        noise_pred = model(x, t_batched)

        alpha = alphas[i]
        alpha_cumprod = alphas_cumprod[i]
        sqrt_one_minus_abar = torch.sqrt(1. - alpha_cumprod)

        mean = (x - (1 - alpha) / sqrt_one_minus_abar * noise_pred) / torch.sqrt(alpha)

        if i > 0:
            noise = torch.randn_like(x)
            variance = torch.sqrt(1. - alpha)
            x = mean + variance * noise
        else:
            x = mean
    model.train()
    return x


# quick untrained sample: should look like pure noise
z = sample(model, n=4, steps=50, device=device)
fig, axes = plt.subplots(1, 4, figsize=(9, 2.4))
for ax, img in zip(axes, z):
    show_img(img, ax)
plt.suptitle('Before training (expected: noise)')
plt.tight_layout()
plt.show()
# Estimated time: ~10-20 s with steps=50; ~5-8 min for full steps=1000

## 5. Training loop

We use AdamW, a cosine LR schedule with linear warmup, mixed precision (AMP), gradient clipping, and an Exponential Moving Average (EMA) of weights for stable sampling.

**Expected time (single T4, batch 64, 64x64 images):**
CelebA-HQ has ~30k images -> **~469 steps/epoch**. At roughly **0.10-0.15 s/step**, one epoch takes **~45 s - 1.2 min**, so all **20 epochs ~15-25 min** (plus the periodic sample grids). If that is too slow, drop `cfg.epochs` to 10 or use `steps=50` for the periodic grids.

In [ ]:
# ---- EMA utilities ----
@torch.no_grad()
def ema_update(model, ema_model, decay=0.9999):
    for p, ema_p in zip(model.parameters(), ema_model.parameters()):
        ema_p.mul_(decay).add_(p, alpha=1 - decay)


@torch.no_grad()
def ema_copy(model):
    ema = DiT().to(device)
    ema.load_state_dict(model.state_dict())
    return ema

In [ ]:
def draw_grid(model_, n_cols=4, n_rows=4, show=True, title='', device=device, steps=cfg.num_timesteps):
    """Generate, tile, and optionally display a grid of samples."""
    model_.eval()
    imgs = sample(model_, n=n_cols * n_rows, steps=steps, device=device)
    imgs = imgs.clamp(-1, 1)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.5, n_rows * 1.5))
    for ax, img in zip(axes.flat, imgs):
        show_img(img, ax)
    if title:
        plt.suptitle(title)
    plt.tight_layout()
    if show:
        plt.show()
    model_.train()
    return imgs

In [ ]:
# ---- Optimizer, scheduler, EMA setup ----
iterations_per_epoch = len(train_dl)
total_steps = iterations_per_epoch * cfg.epochs

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda step: min(1.0, (step + 1) / cfg.warmup_steps) *
                          (0.5 * (1.0 + math.cos(math.pi * min(1.0, step / total_steps))))
)

ema_model = ema_copy(model)
scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp and torch.cuda.is_available())

global_step = 0
print('Steps per epoch:', iterations_per_epoch, '| total steps:', total_steps)
# Estimated time: <1 s

In [ ]:
# ---- Training ----
model.train()
running = 0.0
loss_history = []
t0 = time.time()

for epoch in range(1, cfg.epochs + 1):
    epoch_loss = 0.0
    for x0, _ in train_dl:
        x0 = x0.to(device)
        t = torch.randint(0, cfg.num_timesteps, (x0.size(0),), device=device)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=cfg.use_amp and torch.cuda.is_available()):
            loss = p_losses(model, x0, t, use_amp=cfg.use_amp and torch.cuda.is_available())
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step(global_step)

        ema_update(model, ema_model)

        loss_val = loss.item()
        epoch_loss += loss_val
        running += loss_val
        global_step += 1

        if global_step % cfg.log_every == 0:
            elapsed = time.time() - t0
            it_s = global_step / elapsed
            lr_now = optimizer.param_groups[0]['lr']
            print(f'ep {epoch} it {global_step:5d}  loss {running / cfg.log_every:.4f}'
                  f'  {it_s:.1f} it/s  lr {lr_now:.2e}')
            running = 0.0

    avg = epoch_loss / iterations_per_epoch
    loss_history.append(avg)
    print(f'--- epoch {epoch}/{cfg.epochs} avg loss {avg:.4f} | '
          f'lr {optimizer.param_groups[0]["lr"]:.2e} ---')

    if epoch % cfg.sample_every_n_epochs == 0 or epoch == 1:
        draw_grid(ema_model, n_cols=4, n_rows=4,
                  title=f'EMA samples after epoch {epoch}')

print('Total training time: %.1f s (%.1f min)' % (time.time() - t0, (time.time() - t0) / 60))

## 6. Results

The training curve and a final grid of generated color images. Run the two cells below after training to inspect quality.

**Expected time:** loss curve renders instantly; the 8x8 grid with full `steps=1000` takes ~1-2 min (about ~2-6 s per 100 steps for batch 64).

In [ ]:
# ---- Training loss curve ----
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(loss_history) + 1), loss_history, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Avg training loss')
plt.title('DiT denoising loss (CelebA-HQ)')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# ---- Generate an 8x8 grid from the EMA model (full 1000 DDPM steps) ----
grid = draw_grid(ema_model, n_cols=8, n_rows=8,
                 title='Generated CelebA-HQ faces (EMA model)')
print('Generated', grid.shape[0], 'images.')
# Estimated time: ~1-2 min on T4

## Next steps

- Add **attribute/class conditioning** (smile, gender, age, style) by feeding a label embedding into the adaLN modulation, and enable **classifier-free guidance** for controllable synthesis.
- Scale `hidden_size`, `depth`, and `n_heads` together (DiT-S/B/L/XL) and train longer for higher-fidelity faces.
- Raise `cfg.resolution` to 128 and train with a bigger model — CelebA-HQ scales cleanly up to 256x256.
- Switch to a **DDIM** sampler to cut reverse steps from 1000 to ~50 for a large sampling speed-up.